In [23]:
import torch
import torchvision
import pandas as pd

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("\nDispositivo disponible:")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU no disponible; se utilizará CPU.")

print("\nCUDA:", torch.cuda.is_available())
print("Número de CPU:", __import__("os").cpu_count())

PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu

Dispositivo disponible:
GPU no disponible; se utilizará CPU.

CUDA: False
Número de CPU: 4


In [24]:
from pathlib import Path
import sys

PROJECT_SRC = Path("/kaggle/input/datasets/claudiobances/mlscript")
MANIFEST_PATH = Path("/kaggle/working/reports/dataset_manifest.csv")

sys.path.insert(0, str(PROJECT_SRC))

from dataset import create_dataset

train_dataset = create_dataset(MANIFEST_PATH, "train")
val_dataset = create_dataset(MANIFEST_PATH, "val")
test_dataset = create_dataset(MANIFEST_PATH, "test")

print("Entrenamiento:", len(train_dataset))
print("Validación:", len(val_dataset))
print("Prueba:", len(test_dataset))

image, label = train_dataset[0]

print("\nForma de imagen:", image.shape)
print("Etiqueta:", label)
print("Tipo:", image.dtype)

Entrenamiento: 700
Validación: 150
Prueba: 150

Forma de imagen: torch.Size([3, 224, 224])
Etiqueta: 1
Tipo: torch.float32


In [25]:
import torch
import torch.nn as nn
from torchvision import models

# Configuración
DEVICE = torch.device("cpu")
NUM_CLASSES = 2

# Cargar ResNet18 preentrenada
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Congelar todas las capas
for param in model.parameters():
    param.requires_grad = False

# Reemplazar clasificador final
in_features = model.fc.in_features

model.fc = nn.Linear(
    in_features,
    NUM_CLASSES
)

# Enviar modelo al dispositivo
model = model.to(DEVICE)

# Comprobar parámetros entrenables
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print("Modelo:", model.__class__.__name__)
print("Clases:", NUM_CLASSES)
print("Dispositivo:", DEVICE)
print("Parámetros entrenables:", trainable_params)
print("Parámetros totales:", total_params)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


Modelo: ResNet
Clases: 2
Dispositivo: cpu
Parámetros entrenables: 1026
Parámetros totales: 11177538


In [27]:
import sys
import importlib
from pathlib import Path

PROJECT_SRC = Path("/kaggle/input/datasets/claudiobances/mlscripts1")

if str(PROJECT_SRC) not in sys.path:
    sys.path.insert(0, str(PROJECT_SRC))

import dataset
importlib.reload(dataset)

from dataset import create_dataset

MANIFEST_PATH = Path("/kaggle/working/reports/dataset_manifest.csv")

train_dataset = create_dataset(MANIFEST_PATH, "train")
val_dataset = create_dataset(MANIFEST_PATH, "val")
test_dataset = create_dataset(MANIFEST_PATH, "test")

image, label = train_dataset[0]

print("Ruta del módulo:", dataset.__file__)
print("Forma:", image.shape)
print("Etiqueta:", label)
print("Tipo:", image.dtype)
print("Media aproximada:", image.mean().item())

Ruta del módulo: /kaggle/input/datasets/claudiobances/mlscripts1/dataset.py
Forma: torch.Size([3, 224, 224])
Etiqueta: 1
Tipo: torch.float32
Media aproximada: 1.334212303161621


In [28]:
from torch.utils.data import DataLoader

BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

# Comprobar un lote
images, labels = next(iter(train_loader))

print("Forma de imágenes:", images.shape)
print("Forma de etiquetas:", labels.shape)
print("Etiquetas del lote:", labels)
print("Lotes de entrenamiento:", len(train_loader))
print("Lotes de validación:", len(val_loader))
print("Lotes de prueba:", len(test_loader))

Forma de imágenes: torch.Size([16, 3, 224, 224])
Forma de etiquetas: torch.Size([16])
Etiquetas del lote: tensor([1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0])
Lotes de entrenamiento: 44
Lotes de validación: 10
Lotes de prueba: 10


In [29]:
import torch.nn as nn
import torch.optim as optim

# Función de pérdida
criterion = nn.CrossEntropyLoss()

# Optimizador: únicamente parámetros entrenables
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

# Configuración
EPOCHS = 5
LEARNING_RATE = 0.001

print("Configuración del entrenamiento")
print("Épocas:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Optimizador: Adam")
print("Función de pérdida: CrossEntropyLoss")
print("Parámetros optimizados:", trainable_params)

Configuración del entrenamiento
Épocas: 5
Learning rate: 0.001
Optimizador: Adam
Función de pérdida: CrossEntropyLoss
Parámetros optimizados: 1026


In [30]:
import time

history = {
    "train_loss": [],
    "train_accuracy": []
}

for epoch in range(EPOCHS):

    start_time = time.time()

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    history["train_loss"].append(epoch_loss)
    history["train_accuracy"].append(epoch_accuracy)

    elapsed = time.time() - start_time

    print(
        f"Época [{epoch + 1}/{EPOCHS}] | "
        f"Loss: {epoch_loss:.4f} | "
        f"Accuracy: {epoch_accuracy:.4f} | "
        f"Tiempo: {elapsed:.2f}s"
    )

print("\nENTRENAMIENTO COMPLETADO")

Época [1/5] | Loss: 0.3870 | Accuracy: 0.8500 | Tiempo: 163.08s
Época [2/5] | Loss: 0.2286 | Accuracy: 0.9229 | Tiempo: 161.20s
Época [3/5] | Loss: 0.2156 | Accuracy: 0.9186 | Tiempo: 161.32s
Época [4/5] | Loss: 0.1686 | Accuracy: 0.9386 | Tiempo: 163.92s
Época [5/5] | Loss: 0.2232 | Accuracy: 0.9186 | Tiempo: 162.76s

ENTRENAMIENTO COMPLETADO


In [31]:
model.eval()

val_loss = 0.0
correct = 0
total = 0

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

val_loss /= total
val_accuracy = correct / total

print("RESULTADOS DE VALIDACIÓN")
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
print("Total de imágenes evaluadas:", total)

RESULTADOS DE VALIDACIÓN
Validation Loss: 0.1035
Validation Accuracy: 98.00%
Total de imágenes evaluadas: 150


In [32]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=[0, 1]
)

cm_df = pd.DataFrame(
    cm,
    index=["Real: Roya", "Real: Sanas"],
    columns=["Predicción: Roya", "Predicción: Sanas"]
)

print("MATRIZ DE CONFUSIÓN")
display(cm_df)

print("\nREPORTE DE CLASIFICACIÓN")
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=[0, 1],
        target_names=["Roya", "Sanas"],
        digits=4
    )
)

MATRIZ DE CONFUSIÓN


,Predicción: Roya,Predicción: Sanas
Real: Roya,74,1
Real: Sanas,2,73



REPORTE DE CLASIFICACIÓN
              precision    recall  f1-score   support

        Roya     0.9737    0.9867    0.9801        75
       Sanas     0.9865    0.9733    0.9799        75

    accuracy                         0.9800       150
   macro avg     0.9801    0.9800    0.9800       150
weighted avg     0.9801    0.9800    0.9800       150



In [33]:
from pathlib import Path
import torch

MODEL_DIR = Path("/kaggle/working/models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_name": "resnet18",
    "model_state_dict": model.state_dict(),
    "num_classes": NUM_CLASSES,
    "class_to_idx": {
        "Roya": 0,
        "Sanas": 1
    },
    "image_size": 224,
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "training_config": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "optimizer": "Adam",
        "loss": "CrossEntropyLoss"
    },
    "validation_metrics": {
        "loss": val_loss,
        "accuracy": val_accuracy
    }
}

MODEL_PATH = MODEL_DIR / "resnet18_coffee_base.pth"

torch.save(checkpoint, MODEL_PATH)

print("Modelo guardado correctamente:")
print(MODEL_PATH)
print(f"Tamaño: {MODEL_PATH.stat().st_size / (1024**2):.2f} MB")

Modelo guardado correctamente:
/kaggle/working/models/resnet18_coffee_base.pth
Tamaño: 42.71 MB


In [34]:
from torchvision import models
import torch.nn as nn

# Crear nuevamente la arquitectura
loaded_model = models.resnet18(weights=None)
loaded_model.fc = nn.Linear(
    loaded_model.fc.in_features,
    NUM_CLASSES
)

# Cargar los pesos guardados
checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)

loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model = loaded_model.to(DEVICE)
loaded_model.eval()

# Probar una predicción
sample_image, sample_label = val_dataset[0]

with torch.no_grad():
    output = loaded_model(sample_image.unsqueeze(0).to(DEVICE))
    prediction = torch.argmax(output, dim=1).item()

idx_to_class = {0: "Roya", 1: "Sanas"}

print("Modelo cargado correctamente.")
print("Clase real:", idx_to_class[sample_label])
print("Clase predicha:", idx_to_class[prediction])
print("Arquitectura:", checkpoint["model_name"])

Modelo cargado correctamente.
Clase real: Roya
Clase predicha: Roya
Arquitectura: resnet18
